# Purpose

Prove the P2B foundation operators activation is bounded, deterministic, and complete.

## Lemma: L73

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()

# --- Load authoritative registries ---
agents = yaml.safe_load((repo_root / 'registry' / 'agents.yaml').read_text(encoding='utf-8'))
persona_reg = yaml.safe_load((repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8'))
cap_map = yaml.safe_load((repo_root / 'registry' / 'persona_capability_map.yaml').read_text(encoding='utf-8'))
routing = yaml.safe_load((repo_root / 'registry' / 'executor_routing_v2.yaml').read_text(encoding='utf-8'))

FOUNDATION = [
    ('backend-architect', 13),
    ('devops-automator', 10),
    ('api-tester', 8),
    ('analytics-reporter', 9),
    ('project-shipper', 9),
    ('support-responder', 8),
]

# Build lookup tables
agents_by_name = {a['name']: a for a in agents['agents']}
personas_by_id = {p['persona_id']: p for p in persona_reg['personas']}
cap_by_id = {p['persona_id']: p for p in cap_map['personas']}

# --- Verify each foundation persona ---
for persona_id, expected_actions in FOUNDATION:
    # 1. Registry-backed
    p = personas_by_id[persona_id]
    assert p['coverage_status'] == 'registry-backed', f'{persona_id} not registry-backed'
    assert p['status'] == 'active', f'{persona_id} not active'
    # 2. Action-surfaced with correct count
    a = agents_by_name[persona_id]
    assert len(a['allowed_actions']) == expected_actions, f'{persona_id} action count mismatch'
    # 3. Capability map parity
    c = cap_by_id[persona_id]
    assert c['coverage_status'] == 'registry-backed', f'{persona_id} cap map not registry-backed'

# --- Verify total new actions = 57 ---
total = sum(n for _, n in FOUNDATION)
assert total == 57, f'Expected 57 total actions, got {total}'

# --- Verify remaining personas stay contract-only ---
activated_ids = {pid for pid, _ in FOUNDATION}
# Also exclude the 4 pre-existing registry-backed personas
contract_only = [p for p in persona_reg['personas']
                 if p['persona_id'] not in activated_ids
                 and p.get('coverage_status') == 'persona-contract-only']
assert len(contract_only) == 29, f'Expected 29 contract-only, got {len(contract_only)}'

# --- Verify department packs exist ---
for dept in ['engineering', 'testing', 'studio_operations', 'project_management']:
    path = repo_root / 'registry' / f'department_pack_{dept}_v1.yaml'
    assert path.exists(), f'Missing department pack: {path}'

# --- Verify routing coverage ---
assert (repo_root / 'registry' / 'executor_routing_v2.yaml').exists()

print('ALL L73 CHECKS PASSED')